In [1]:
pip install mediapipe opencv-python tensorflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: C:\Users\Dizer\AppData\Local\Programs\Python\Python39\python.exe -m pip install --upgrade pip


In [2]:
!pip install scikit-learn

You should consider upgrading via the 'C:\Users\Dizer\OneDrive\Desktop\x\venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [3]:
import cv2
import numpy as np
import mediapipe as mp
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.efficientnet import preprocess_input
from sklearn.preprocessing import LabelEncoder
import time

# Load the trained model (ensure to change the path to your trained model)
model = load_model('asl_model_efficientnetb000.h5')

# Load Label Encoder for class names
label_encoder = LabelEncoder()
class_names = ['again', 'bad', 'bathroom', 'book', 'busy', 'do not want', 'eat', 'father', 'fine', 'finish',
               'forget', 'go', 'good', 'happy', 'hello', 'help', 'how', 'i', 'learn', 'like', 'meet', 'milk',
               'more', 'mother', 'my', 'name', 'need', 'nice', 'no', 'please', 'question', 'right', 'sad',
               'same', 'see you later', 'thank you', 'want', 'what', 'when', 'where', 'which', 'who', 'why',
               'wrong', 'yes', 'you', 'your']
label_encoder.fit(class_names)

# Set up MediaPipe Hands
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(min_detection_confidence=0.5, min_tracking_confidence=0.5)
mp_draw = mp.solutions.drawing_utils

# Define video capture
cap = cv2.VideoCapture(0)  # 0 is the default camera

# Set up the frame size for the capture
frame_width = 224  # Input size for EfficientNet
frame_height = 224
target_frames = 12  # Number of frames expected by the model
frame_sequence = []

# Variables to control prediction display timeout
predicted_class = None
prediction_time = None
prediction_display_time = 2  # Time to display prediction in seconds

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Flip the frame horizontally for a more intuitive experience
    frame = cv2.flip(frame, 1)

    # Convert the frame to RGB for MediaPipe
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Process the frame for hand landmarks
    results = hands.process(rgb_frame)

    # Draw landmarks on hand if detected
    if results.multi_hand_landmarks:
        for landmarks in results.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, landmarks, mp_hands.HAND_CONNECTIONS)

    # Resize the frame for model input
    resized_frame = cv2.resize(frame, (frame_width, frame_height))
    frame_sequence.append(resized_frame)

    # If we have collected enough frames, preprocess and predict
    if len(frame_sequence) == target_frames:
        # Convert the sequence to a numpy array and preprocess
        frame_sequence_array = np.array(frame_sequence)
        frame_sequence_array = np.expand_dims(frame_sequence_array, axis=0)  # Add batch dimension
        frame_sequence_array = preprocess_input(frame_sequence_array)

        # Prediction
        predictions = model.predict(frame_sequence_array)
        predicted_class = label_encoder.inverse_transform([np.argmax(predictions)])

        # Record the time when prediction is made
        prediction_time = time.time()

        # Reset frame sequence after prediction
        frame_sequence = []

    # If enough time has passed, display the predicted class
    if predicted_class is not None and (time.time() - prediction_time) < prediction_display_time:
        cv2.putText(frame, f'Predicted: {predicted_class[0]}', (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    else:
        predicted_class = None  # Reset the prediction after the display time

    # Display the frame with landmarks and prediction
    cv2.imshow('Real-Time Hand Gesture Recognition', frame)

    # Break the loop if 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the capture and close windows
cap.release()
cv2.destroyAllWindows()


TypeError: Error when deserializing class 'InputLayer' using config={'batch_shape': [None, 12, 224, 224, 3], 'dtype': 'float32', 'sparse': False, 'name': 'input_layer'}.

Exception encountered: Unrecognized keyword arguments: ['batch_shape']